In [1]:
#Gauntlet #6 – The Time Series Challenge

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')

departure = pd.to_datetime('1912-04-15 18:00:00')
np.random.seed(42)

df['Booking_DateTime'] = departure - pd.Timedelta(days=30) + pd.to_timedelta(np.random.randint(-240, 240, size=len(df)), unit='H')
df['Booking_DateTime'] = df['Booking_DateTime'] + pd.to_timedelta(np.random.randint(0, 60, size=len(df)), unit='T')

C:\Users\Faizan Toheed\AppData\Local\Temp\ipykernel_5376\2889920707.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  df['Booking_DateTime'] = departure - pd.Timedelta(days=30) + pd.to_timedelta(np.random.randint(-240, 240, size=len(df)), unit='H')
C:\Users\Faizan Toheed\AppData\Local\Temp\ipykernel_5376\2889920707.py:10: FutureWarning: 'T' is deprecated and will be removed in a future version. Please use 'min' instead of 'T'.
  df['Booking_DateTime'] = df['Booking_DateTime'] + pd.to_timedelta(np.random.randint(0, 60, size=len(df)), unit='T')


In [3]:
# Task 1: Localize to London, then convert to UTC
df['Booking_DateTime'] = df['Booking_DateTime'].dt.tz_localize('Europe/London')
df['Booking_UTC'] = df['Booking_DateTime'].dt.tz_convert('UTC')

print("=== Task 1: Localized Timezone Info ===")
print(df['Booking_DateTime'].head())
print("\nTimezone:", df['Booking_DateTime'].dt.tz)
print("\nConverted to UTC:")
print(df['Booking_UTC'].head())

=== Task 1: Localized Timezone Info ===
0   1912-03-11 00:30:00+00:00
1   1912-03-24 21:00:00+00:00
2   1912-03-21 06:52:00+00:00
3   1912-03-18 00:53:00+00:00
4   1912-03-11 04:53:00+00:00
Name: Booking_DateTime, dtype: datetime64[ns, Europe/London]

Timezone: Europe/London

Converted to UTC:
0   1912-03-11 00:30:00+00:00
1   1912-03-24 21:00:00+00:00
2   1912-03-21 06:52:00+00:00
3   1912-03-18 00:53:00+00:00
4   1912-03-11 04:53:00+00:00
Name: Booking_UTC, dtype: datetime64[ns, UTC]


In [4]:
# Task 2: Extract components using .dt
df['Booking_Month'] = df['Booking_DateTime'].dt.month
df['Booking_Hour'] = df['Booking_DateTime'].dt.hour
df['Booking_Weekday_Name'] = df['Booking_DateTime'].dt.day_name()

print("\n=== Task 2: Extracted Components ===")
print(df[['Booking_DateTime', 'Booking_Month', 'Booking_Hour', 'Booking_Weekday_Name']].head())


=== Task 2: Extracted Components ===
           Booking_DateTime  Booking_Month  Booking_Hour Booking_Weekday_Name
0 1912-03-11 00:30:00+00:00              3             0               Monday
1 1912-03-24 21:00:00+00:00              3            21               Sunday
2 1912-03-21 06:52:00+00:00              3             6             Thursday
3 1912-03-18 00:53:00+00:00              3             0               Monday
4 1912-03-11 04:53:00+00:00              3             4               Monday


In [5]:
# Departure time (London timezone)
departure_dt = pd.to_datetime('1912-04-15 18:00:00').tz_localize('Europe/London')

# Task 3: Calculate exact hours difference
df['Hours_Before'] = (departure_dt - df['Booking_DateTime']) / pd.Timedelta(hours=1)

print("\n=== Task 3: Hours Before Departure ===")
print(df[['Booking_DateTime', 'Hours_Before']].head())


=== Task 3: Hours Before Departure ===
           Booking_DateTime  Hours_Before
0 1912-03-11 00:30:00+00:00    857.500000
1 1912-03-24 21:00:00+00:00    525.000000
2 1912-03-21 06:52:00+00:00    611.133333
3 1912-03-18 00:53:00+00:00    689.116667
4 1912-03-11 04:53:00+00:00    853.116667


In [6]:
# Task 4: Set index to localized Booking_DateTime and resample weekly
df_indexed = df.set_index('Booking_DateTime')

weekly = df_indexed.resample('W').agg({
    'Fare': 'sum',
    'PassengerId': 'nunique'
}).rename(columns={'Fare': 'Total_Fare', 'PassengerId': 'Unique_Passengers'})

print("\n=== Task 4: Weekly Aggregation ===")
print(weekly.head())


=== Task 4: Weekly Aggregation ===
                           Total_Fare  Unique_Passengers
Booking_DateTime                                        
1912-03-10 00:00:00+00:00   4417.7204                173
1912-03-17 00:00:00+00:00  12950.7538                336
1912-03-24 00:00:00+00:00   8671.9124                298
1912-03-31 00:00:00+00:00   2653.5627                 84


In [7]:
# Task 5: 2-week rolling average on passenger counts (no centering)
weekly['Rolling_Avg_Passengers'] = weekly['Unique_Passengers'].rolling(window=2).mean()

# Find the week with the highest rolling average
max_rolling_week = weekly['Rolling_Avg_Passengers'].idxmax()
max_rolling_value = weekly.loc[max_rolling_week, 'Rolling_Avg_Passengers']

print("\n=== Task 5: 2-Week Rolling Average ===")
print(weekly[['Unique_Passengers', 'Rolling_Avg_Passengers']].head())
print(f"\nHighest rolling average week ending: {max_rolling_week.date()}, Value: {max_rolling_value:.2f}")


=== Task 5: 2-Week Rolling Average ===
                           Unique_Passengers  Rolling_Avg_Passengers
Booking_DateTime                                                    
1912-03-10 00:00:00+00:00                173                     NaN
1912-03-17 00:00:00+00:00                336                   254.5
1912-03-24 00:00:00+00:00                298                   317.0
1912-03-31 00:00:00+00:00                 84                   191.0

Highest rolling average week ending: 1912-03-24, Value: 317.00


In [8]:
# Task 6: Create column for previous week's passenger count
weekly['Prev_Week_Passengers'] = weekly['Unique_Passengers'].shift(1)

print("\n=== Task 6: Previous Week Passengers ===")
print(weekly[['Unique_Passengers', 'Prev_Week_Passengers']].head())


=== Task 6: Previous Week Passengers ===
                           Unique_Passengers  Prev_Week_Passengers
Booking_DateTime                                                  
1912-03-10 00:00:00+00:00                173                   NaN
1912-03-17 00:00:00+00:00                336                 173.0
1912-03-24 00:00:00+00:00                298                 336.0
1912-03-31 00:00:00+00:00                 84                 298.0


In [9]:
# Task 7: Convert first 5 rows of UTC to Asia/Karachi
df['Booking_Karachi'] = df['Booking_UTC'].dt.tz_convert('Asia/Karachi')

print("\n=== Task 7: UTC vs Karachi Time (First 5 Rows) ===")
print(df[['Booking_UTC', 'Booking_Karachi']].head())


=== Task 7: UTC vs Karachi Time (First 5 Rows) ===
                Booking_UTC           Booking_Karachi
0 1912-03-11 00:30:00+00:00 1912-03-11 06:00:00+05:30
1 1912-03-24 21:00:00+00:00 1912-03-25 02:30:00+05:30
2 1912-03-21 06:52:00+00:00 1912-03-21 12:22:00+05:30
3 1912-03-18 00:53:00+00:00 1912-03-18 06:23:00+05:30
4 1912-03-11 04:53:00+00:00 1912-03-11 10:23:00+05:30


In [10]:
# Task 8: Generate all Fridays between earliest and latest booking dates
earliest = df['Booking_DateTime'].min().normalize()
latest   = df['Booking_DateTime'].max().normalize()

fridays = pd.date_range(start=earliest, end=latest, freq='W-FRI')
print("\n=== Task 8: All Fridays in Booking Period ===")
print(fridays)
print(f"Total Fridays: {len(fridays)}")


=== Task 8: All Fridays in Booking Period ===
DatetimeIndex(['1912-03-08 00:00:00+00:00', '1912-03-15 00:00:00+00:00',
               '1912-03-22 00:00:00+00:00'],
              dtype='datetime64[ns, Europe/London]', freq='W-FRI')
Total Fridays: 3
